# Análisis de datos

## Código sacado de offline_eval.ipynb

In [ ]:
import os
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import requests

for line in Path("../.env").read_text().splitlines() if Path("../.env").exists() else []:
    key, _, value = line.partition("=")
    os.environ.setdefault(key.strip(), value.strip())
API = os.environ.get("RAWTREE_API_URL", "https://api.rawtree.com")
http = requests.Session()
http.headers["Authorization"] = f"Bearer {os.environ['RAWTREE_API_KEY']}"

def sql(q) -> pd.DataFrame:
    r = http.post(f"{API}/v1/query", json={"sql": q}, timeout=600)
    r.raise_for_status()
    return pd.DataFrame(r.json()["data"])

Ejemplo:

In [2]:
last_day = sql("SELECT toDate(max(ts::DateTime64(3))) AS d FROM shop.history_sessions").d[0]
CUT = str((pd.Timestamp(last_day) - pd.Timedelta(days=7)).date())   # 'YYYY-MM-DD'
print("train < ", CUT, "<= test")

# Columns are Dynamic (schemaless): filters work as is; joins and some aggregates need casts like ::String
test = sql(f"""
    SELECT request_id, ts, user_id, context, recent_views, shown, propensity, clicked, purchased
    FROM shop.history_sessions WHERE ts >= '{CUT}' ORDER BY rand() LIMIT 100000""")
test["clicked"] = test.clicked.replace("", None)
print(len(test), "test impressions, observed CTR", round(test.clicked.notna().mean(), 4))
test.head(3)

train <  2026-11-07 <= test
100000 test impressions, observed CTR 0.0383


,request_id,ts,user_id,context,recent_views,shown,propensity,clicked,purchased
0,r_ee27a18380ed096b,2026-11-10 03:05:49.533000000,u_782a47c0,"{'device': 'android', 'page': 'product', 'regi...","[{'product_id': 'p_8914eb', 'ts': '2026-11-10 ...","[p_208432, p_adc3e6, p_f391d6, p_08b1a0, p_09a...","[0.001163, 0.268949, 0.000237, 0.008724, 0.156...",NaN,NaN
1,r_eefcd62e62a73c07,2026-11-13 03:34:26.883000000,u_95384ed4,"{'device': 'ios', 'page': 'product', 'region':...","[{'product_id': 'p_93aeec', 'ts': '2026-11-13 ...","[p_82afd0, p_5639e2, p_fcf075, p_65b5a3, p_664...","[0.003962, 0.029644, 0.003122, 0.004293, 0.002...",p_52e7e7,p_52e7e7
2,r_7fc058a4fedc2182,2026-11-13 03:54:42.765000000,u_c83e4f46,"{'device': 'android', 'page': 'category', 'reg...","[{'product_id': 'p_370ec7', 'ts': '2026-11-13 ...","[p_5733ef, p_9f6f74, p_7ca398, p_bc0984, p_2f0...","[0.368928, 0.016695, 0.089816, 0.029263, 0.179...",NaN,NaN


## Más datos

Nota: podemos ver todas las tablas porque hemos empezado a hacer submissions con el baseline (guía en el README).

### Consulta equivalente a .head() de pandas en las diferentes tablas

In [9]:
def query_head(tabla: str, n: int = 5) -> pd.DataFrame:
    """Equivalente a pd.head con SQL."""
    query = f"SELECT * from {tabla} LIMIT {n}"
    return sql(query)

#### Available from the start

In [11]:
query_head("shop.catalog")

,brand,category,created_at,description,is_active,price_eur,product_id,subcategory
0,Vedo,Sports & Outdoors,2026-01-21 07:00:00.000000000,Vedo Yoga Mats with organic ingredients and ex...,True,14.99,p_001897,Yoga Mats
1,Zore,Fashion,2025-11-06 07:00:00.000000000,"From Zore (Jackets): wireless connectivity, a ...",True,4.99,p_00196b,Jackets
2,Sanine,Electronics,2025-01-28 07:00:00.000000000,Sanine Laptop Sleeves with a luxury feel and a...,True,57.99,p_001a20,Laptop Sleeves
3,Vebexa,Sports & Outdoors,2024-09-18 07:00:00.000000000,Vebexa Running Shoes with built for profession...,True,29.99,p_001d7f,Running Shoes
4,Nibe,Home & Kitchen,2025-10-15 07:00:00.000000000,Nibe Cutting Boards with a minimalist design a...,True,26.99,p_0032c5,Cutting Boards


In [ ]:
query_head("shop.history_sessions")

,clicked,context.device,context.page,context.region,propensity,purchased,recent_views,request_id,revenue_eur,shown,ts,user_id
0,None,web,product,Catalunya,"[0.218886, 0.002833, 0.11516, 0.321647, 0.0292...",None,"[{'product_id': 'p_159190', 'ts': '2026-09-15 ...",r_50309edc92778291,0,"[p_8ca8ed, p_8ac75a, p_e2c959, p_2c88d5, p_8fb...",2026-09-15 07:00:01.456000000,u_20cbc5d5
1,None,web,product,Madrid,"[0.032807, 0.011809, 0.008224, 0.025852, 0.290...",None,"[{'product_id': 'p_f3c09e', 'ts': '2026-09-15 ...",r_7a441a0fb217005e,0,"[p_032d24, p_40bbb4, p_66f831, p_d245de, p_adc...",2026-09-15 07:00:13.975000000,u_cd609f53
2,None,android,category,Aragón,"[0.009835, 0.271305, 0.005089, 0.050632, 0.165...",None,"[{'product_id': 'p_f41d0b', 'ts': '2026-09-15 ...",r_7fa58169a04e436c,0,"[p_9c2b8b, p_adc3e6, p_883ab9, p_07b81b, p_09a...",2026-09-15 07:00:19.633000000,u_5a69bbf5
3,None,ios,product,Aragón,"[0.003099, 0.219566, 0.019525, 0.290605, 0.007...",None,"[{'product_id': 'p_07bab2', 'ts': '2026-09-15 ...",r_28800da566ece94b,0,"[p_fe5a63, p_8ca8ed, p_b8227f, p_2c88d5, p_819...",2026-09-15 07:00:22.418000000,u_039914dd
4,None,android,product,Catalunya,"[0.004059, 0.219778, 0.285283, 0.16151, 0.0326...",None,"[{'product_id': 'p_f67b9c', 'ts': '2026-09-15 ...",r_a3372ac6405c3511,0,"[p_ef8559, p_8ca8ed, p_2c88d5, p_e2c959, p_4ba...",2026-09-15 07:00:47.069000000,u_79d89af1


In [12]:
query_head("shop.history_events")

,device,event_type,price_eur,product_id,region,session_id,ts,user_id
0,android,view,9.99,p_d06b48,Catalunya,s_9aee4d52a08a,2026-09-15 07:00:06.371000000,u_04b42c08
1,web,view,85.99,p_9dc613,Comunitat Valenciana,s_8e7292024915,2026-09-15 07:00:13.592000000,u_9f64d903
2,android,view,215.99,p_134914,Catalunya,s_9aee4d52a08a,2026-09-15 07:00:23.341000000,u_04b42c08
3,web,view,9.99,p_f8920d,Catalunya,s_92b57e4008c4,2026-09-15 07:00:45.091000000,u_ffc15e95
4,web,view,30.99,p_f709f7,Comunitat Valenciana,s_8e7292024915,2026-09-15 07:01:01.182000000,u_9f64d903


#### During the event (Saturday to Sunday)

In [14]:
query_head("shop.requests")

,context.device,context.page,context.region,is_new_user,recent_views,request_id,ts,user_id
0,ios,product,Castilla y León,False,"[{'product_id': 'p_354478', 'ts': '2026-11-14 ...",r_3490d43fc4edca27,2026-11-14 09:00:00.014000000,u_dde95cc2
1,android,product,Murcia,True,"[{'product_id': 'p_5be836', 'ts': '2026-11-14 ...",r_be9a6b921540ee06,2026-11-14 09:00:00.382000000,u_157775db
2,android,home,Andalucía,True,[],r_5a80914f0566f217,2026-11-14 09:00:00.687000000,u_41216ce0
3,ios,product,Catalunya,True,"[{'product_id': 'p_170f13', 'ts': '2026-11-14 ...",r_a9ea6a9e25eb3f6a,2026-11-14 09:00:01.041000000,u_061f799b
4,ios,product,País Vasco,True,"[{'product_id': 'p_32773e', 'ts': '2026-11-14 ...",r_5c385016b1d75ace,2026-11-14 09:00:01.352000000,u_f2fcbfa3


In [15]:
query_head("shop.events")

,device,event_type,price_eur,product_id,region,session_id,ts,user_id
0,ios,view,25.99,p_931b47,Andalucía,s_d15161d3da9d,2026-11-14 09:00:00.014000000,u_a7c85306
1,ios,view,39.99,p_0468c1,Andalucía,s_d15161d3da9d,2026-11-14 09:00:00.021000000,u_a7c85306
2,ios,view,37.99,p_0cb387,Andalucía,s_d15161d3da9d,2026-11-14 09:00:00.028000000,u_a7c85306
3,ios,view,25.99,p_164481,Andalucía,s_d15161d3da9d,2026-11-14 09:00:00.035000000,u_a7c85306
4,web,view,21.99,p_d72d81,Andalucía,s_463986306ad7,2026-11-14 09:00:00.085000000,u_c13fe3d2


In [16]:
query_head("shop.catalog_updates")

,brand,category,change,created_at,description,is_active,price_eur,product_id,subcategory,ts
0,Maka,Sports & Outdoors,out_of_stock,2024-11-05 07:00:00.000000000,Maka Running Shoes with a vintage look and com...,False,11.99,p_66211c,Running Shoes,2026-11-14 09:00:00.000000000
1,Tinove,Sports & Outdoors,out_of_stock,2026-09-12 07:00:00.000000000,Tinove Bicycles with made from recycled materi...,False,48.99,p_8c4c8d,Bicycles,2026-11-14 09:00:00.000000000
2,Minoma,Books & Media,out_of_stock,2025-04-10 07:00:00.000000000,"E-Reader Covers by Minoma, featuring a vintage...",False,256.99,p_5f97a9,E-Reader Covers,2026-11-14 09:00:00.000000000
3,Maka,Sports & Outdoors,out_of_stock,2026-08-19 07:00:00.000000000,"Maka Tents line, known for built for professio...",False,44.99,p_f0bc6c,Tents,2026-11-14 09:05:00.000000000
4,Kofi,Electronics,out_of_stock,2025-07-02 07:00:00.000000000,"From Kofi (Laptop Sleeves): bold colours, a mi...",False,34.99,p_eab680,Laptop Sleeves,2026-11-14 09:05:00.000000000


In [17]:
query_head("team_30.feedback")

,clicked,fallback_reason,latency_ms,purchased,request_id,revenue_eur,served_by,shown,ts
0,,,0.0,,__init__,0,init,[],2026-01-01 00:00:00.000000000
1,,missing,NaN,,r_39adfc7374a28fa8,0,fallback,"[p_0210c5, p_09a938, p_1036d3, p_e2454f, p_2c8...",2026-11-14 11:20:22.247000000
2,,missing,NaN,,r_267e6a225ef7debd,0,fallback,"[p_0210c5, p_09a938, p_1036d3, p_e2454f, p_2c8...",2026-11-14 11:20:22.602000000
3,,missing,NaN,,r_a5d34246899ca29d,0,fallback,"[p_0210c5, p_09a938, p_1036d3, p_e2454f, p_2c8...",2026-11-14 11:20:22.921000000
4,,missing,NaN,,r_c1eead79cc568451,0,fallback,"[p_0210c5, p_09a938, p_1036d3, p_e2454f, p_2c8...",2026-11-14 11:20:23.236000000


Nota: nos interesa ordenar por fecha para tener las últimas.

In [ ]:
sql("")